In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

nav = pd.read_csv("../data/processed/clean_nav.csv")

perf = pd.read_csv(
    "../data/processed/clean_performance.csv"
)

txn = pd.read_csv(
    "../data/processed/clean_transactions.csv"
)

portfolio = pd.read_csv(
    "../data/processed/clean_portfolio_holdings.csv"
)

nav["date"] = pd.to_datetime(nav["date"])

txn["transaction_date"] = pd.to_datetime(
    txn["transaction_date"]
)

In [ ]:
nav = nav.sort_values(
    ["amfi_code","date"]
)

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)

var_results = []

for fund in nav["amfi_code"].unique():

    returns = nav[
        nav["amfi_code"] == fund
    ]["daily_return"].dropna()

    var95 = np.percentile(
        returns,
        5
    )

    cvar95 = returns[
        returns <= var95
    ].mean()

    var_results.append([
        fund,
        var95,
        cvar95
    ])

var_cvar_report = pd.DataFrame(
    var_results,
    columns=[
        "amfi_code",
        "VaR_95",
        "CVaR_95"
    ]
)

var_cvar_report.to_csv(
    "../data/processed/var_cvar_report.csv",
    index=False
)

var_cvar_report.head()

In [3]:
top5 = perf.head(5)["amfi_code"].tolist()

In [ ]:
plt.figure(figsize=(14,6))

for fund in top5:

    temp = nav[
        nav["amfi_code"] == fund
    ].copy()

    rolling_sharpe = (

        temp["daily_return"]
        .rolling(90)
        .mean()

        /

        temp["daily_return"]
        .rolling(90)
        .std()

    ) * np.sqrt(252)

    plt.plot(
        temp["date"],
        rolling_sharpe,
        label=str(fund)
    )

plt.legend()

plt.title(
    "Rolling 90-Day Sharpe Ratio"
)

plt.savefig(
    "../reports/charts/rolling_sharpe_chart.png"
)

plt.show()

In [5]:
first_txn = txn.groupby(
    "investor_id"
)["transaction_date"].min()

cohort = first_txn.dt.year

txn["cohort_year"] = txn[
    "investor_id"
].map(cohort)

In [ ]:
cohort_summary = txn.groupby(
    "cohort_year"
).agg({

    "amount_inr":"mean"

}).reset_index()

cohort_summary

In [7]:
top_fund = (

    txn.groupby(
        ["cohort_year","amfi_code"]
    )

    .size()

    .reset_index(name="count")

)

In [8]:
sip = txn[
    txn["transaction_type"] == "Sip"
].copy()

In [9]:
sip_counts = sip.groupby(
    "investor_id"
).size()

eligible = sip_counts[
    sip_counts >= 6
].index

sip = sip[
    sip["investor_id"].isin(
        eligible
    )
]

In [ ]:
risk_list = []

for inv in sip["investor_id"].unique():

    temp = sip[
        sip["investor_id"] == inv
    ]

    temp = temp.sort_values(
        "transaction_date"
    )

    gaps = (
        temp["transaction_date"]
        .diff()
        .dt.days
    )

    avg_gap = gaps.mean()

    risk_status = (
        "At Risk"
        if avg_gap > 35
        else "Healthy"
    )

    risk_list.append([
        inv,
        avg_gap,
        risk_status
    ])

sip_continuity = pd.DataFrame(
    risk_list,
    columns=[
        "investor_id",
        "avg_gap_days",
        "status"
    ]
)

sip_continuity.head()

In [ ]:
hhi_results = []

for fund in portfolio[
    "amfi_code"
].unique():

    temp = portfolio[
        portfolio["amfi_code"] == fund
    ]

    hhi = np.sum(
        (
            temp["weight_pct"]/100
        ) ** 2
    )

    hhi_results.append([
        fund,
        hhi
    ])

hhi_df = pd.DataFrame(
    hhi_results,
    columns=[
        "amfi_code",
        "HHI"
    ]
)

hhi_df.sort_values(
    "HHI",
    ascending=False
).head()